In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("TradeCorpETL").getOrCreate()

In [2]:
df_categories = spark.read.csv("../data/categories.csv",header=True,inferSchema=True)
df_customers = spark.read.csv("../data/customers.csv",header=True,inferSchema=True)
df_employees = spark.read.csv("../data/employees.csv",header=True,inferSchema=True)
df_order_details = spark.read.csv("../data/order_details.csv",header=True,inferSchema=True)
df_orders = spark.read.csv("../data/orders.csv",header=True,inferSchema=True)
df_products = spark.read.csv("../data/products.csv",header=True,inferSchema=True)
df_shippers = spark.read.csv("../data/shippers.csv",header=True,inferSchema=True)
df_suppliers = spark.read.csv("../data/suppliers.csv",header=True,inferSchema=True)

In [3]:
from pyspark.sql.functions import col

In [ ]:
# Q11

In [10]:
for name,df in [("df_categories",df_categories),
                ("df_customers",df_customers),
                ("df_employees",df_employees),
                ("df_order_details",df_order_details),
                ("df_orders",df_orders),
                ("df_products",df_products),
                ("df_shippers",df_shippers),
                ("df_suppliers",df_suppliers)]:
    for c in df.columns:
        nbn = df.filter(col(c).isNull()).count()
        if nbn > 0:
            print(name,c,nbn)

df_customers region 60
df_customers postal_code 1
df_customers fax 22
df_employees region 4
df_employees reports_to 1
df_orders shipped_date 21
df_orders ship_region 507
df_orders ship_postal_code 19
df_suppliers region 20
df_suppliers fax 16
df_suppliers homepage 24


In [ ]:
# Q12

In [4]:
df_orders_clean = df_orders.filter(col("shipped_date").isNotNull())

In [5]:
med_price = df_products.approxQuantile("unit_price", [0.5], 0)[0]

In [ ]:
# Q13 non applicable

In [6]:
from pyspark.sql.functions import when

In [7]:
df_products_clean = df_products.withColumn("unit_price",when(col("unit_price").isNull(), med_price).otherwise(col("unit_price")))

In [8]:
df_products_clean.filter(col("unit_price").isNull()).count()

0

In [9]:
from pyspark.sql.functions import min

In [10]:
df_products_clean.select(min("unit_price")).show()

+---------------+
|min(unit_price)|
+---------------+
|            2.5|
+---------------+



In [11]:
from pyspark.sql.functions import max

In [12]:
df_products_clean.select(max("unit_price")).show()

+---------------+
|max(unit_price)|
+---------------+
|          263.5|
+---------------+



In [13]:
df_orders.dtypes

[('order_id', 'int'),
 ('customer_id', 'string'),
 ('employee_id', 'int'),
 ('order_date', 'date'),
 ('required_date', 'date'),
 ('shipped_date', 'date'),
 ('ship_via', 'int'),
 ('freight', 'double'),
 ('ship_name', 'string'),
 ('ship_address', 'string'),
 ('ship_city', 'string'),
 ('ship_region', 'string'),
 ('ship_postal_code', 'string'),
 ('ship_country', 'string')]

In [14]:
df_order_details.dtypes

[('order_id', 'int'),
 ('product_id', 'int'),
 ('unit_price', 'double'),
 ('quantity', 'int'),
 ('discount', 'double')]

In [ ]:
# Q14

In [15]:
df_customers_clean = df_customers

In [16]:
from pyspark.sql.functions import trim, initcap, upper

In [17]:
for c,type in df_customers_clean.dtypes:
    if type == 'string':
        df_customers_clean = df_customers_clean.withColumn(c,trim(col(c)))
        if c == "contact_name":
            df_customers_clean = df_customers_clean.withColumn(c,initcap(col(c)))
        elif c == "country":
            df_customers_clean = df_customers_clean.withColumn(c,upper(col(c)))

In [18]:
df_customers_clean.show()

+-----------+--------------------+------------------+--------------------+--------------------+------------+------+-----------+-----------+--------------+--------------+
|customer_id|        company_name|      contact_name|       contact_title|             address|        city|region|postal_code|    country|         phone|           fax|
+-----------+--------------------+------------------+--------------------+--------------------+------------+------+-----------+-----------+--------------+--------------+
|      ALFKI| Alfreds Futterkiste|      Maria Anders|Sales Representative|       Obere Str. 57|      Berlin|  NULL|      12209|    GERMANY|   030-0074321|   030-0076545|
|      ANATR|Ana Trujillo Empa...|      Ana Trujillo|               Owner|Avda. de la Const...| M�xico D.F.|  NULL|      05021|     MEXICO|  (5) 555-4729|  (5) 555-3745|
|      ANTON|Antonio Moreno Ta...|    Antonio Moreno|               Owner|     Mataderos  2312| M�xico D.F.|  NULL|      05023|     MEXICO|  (5) 555-3

In [19]:
df_customers.show()

+-----------+--------------------+------------------+--------------------+--------------------+------------+------+-----------+-----------+--------------+--------------+
|customer_id|        company_name|      contact_name|       contact_title|             address|        city|region|postal_code|    country|         phone|           fax|
+-----------+--------------------+------------------+--------------------+--------------------+------------+------+-----------+-----------+--------------+--------------+
|      ALFKI| Alfreds Futterkiste|      Maria Anders|Sales Representative|       Obere Str. 57|      Berlin|  NULL|      12209|    Germany|   030-0074321|   030-0076545|
|      ANATR|Ana Trujillo Empa...|      Ana Trujillo|               Owner|Avda. de la Const...| M�xico D.F.|  NULL|      05021|     Mexico|  (5) 555-4729|  (5) 555-3745|
|      ANTON|Antonio Moreno Ta...|    Antonio Moreno|               Owner|     Mataderos  2312| M�xico D.F.|  NULL|      05023|     Mexico|  (5) 555-3

In [ ]:
# Q15

In [20]:
df_order_details_clean = df_order_details.withColumnRenamed("unit_price","prix_unitaire").withColumnRenamed("quantity","quantite")

In [21]:
df_order_details_clean.show()

+--------+----------+-------------+--------+--------+
|order_id|product_id|prix_unitaire|quantite|discount|
+--------+----------+-------------+--------+--------+
|   10248|        11|         14.0|      12|     0.0|
|   10248|        42|          9.8|      10|     0.0|
|   10248|        72|         34.8|       5|     0.0|
|   10249|        14|         18.6|       9|     0.0|
|   10249|        51|         42.4|      40|     0.0|
|   10250|        41|          7.7|      10|     0.0|
|   10250|        51|         42.4|      35|    0.15|
|   10250|        65|         16.8|      15|    0.15|
|   10251|        22|         16.8|       6|    0.05|
|   10251|        57|         15.6|      15|    0.05|
|   10251|        65|         16.8|      20|     0.0|
|   10252|        20|         64.8|      40|    0.05|
|   10252|        33|          2.0|      25|    0.05|
|   10252|        60|         27.2|      40|     0.0|
|   10253|        31|         10.0|      20|     0.0|
|   10253|        39|       

In [22]:
df_orders_clean = df_orders.withColumnRenamed("ship_via","shipper_id")

In [23]:
df_orders_clean.show(2)

+--------+-----------+-----------+----------+-------------+------------+----------+-------+--------------------+------------------+---------+-----------+----------------+------------+
|order_id|customer_id|employee_id|order_date|required_date|shipped_date|shipper_id|freight|           ship_name|      ship_address|ship_city|ship_region|ship_postal_code|ship_country|
+--------+-----------+-----------+----------+-------------+------------+----------+-------+--------------------+------------------+---------+-----------+----------------+------------+
|   10248|      VINET|          5|1996-07-04|   1996-08-01|  1996-07-16|         3|  32.38|Vins et alcools C...|59 rue de l'Abbaye|    Reims|       NULL|           51100|      France|
|   10249|      TOMSP|          6|1996-07-05|   1996-08-16|  1996-07-10|         1|  11.61|  Toms Spezialit�ten|     Luisenstr. 48|  M�nster|       NULL|           44087|     Germany|
+--------+-----------+-----------+----------+-------------+------------+--------

In [ ]:
# Q16

In [24]:
from pyspark.sql.functions import round

In [25]:
df_order_details_clean = df_order_details_clean.withColumn("sous_total",round(col("prix_unitaire")*col("quantite")*(1-col("discount")),2))

In [26]:
df_order_details_clean.show()

+--------+----------+-------------+--------+--------+----------+
|order_id|product_id|prix_unitaire|quantite|discount|sous_total|
+--------+----------+-------------+--------+--------+----------+
|   10248|        11|         14.0|      12|     0.0|     168.0|
|   10248|        42|          9.8|      10|     0.0|      98.0|
|   10248|        72|         34.8|       5|     0.0|     174.0|
|   10249|        14|         18.6|       9|     0.0|     167.4|
|   10249|        51|         42.4|      40|     0.0|    1696.0|
|   10250|        41|          7.7|      10|     0.0|      77.0|
|   10250|        51|         42.4|      35|    0.15|    1261.4|
|   10250|        65|         16.8|      15|    0.15|     214.2|
|   10251|        22|         16.8|       6|    0.05|     95.76|
|   10251|        57|         15.6|      15|    0.05|     222.3|
|   10251|        65|         16.8|      20|     0.0|     336.0|
|   10252|        20|         64.8|      40|    0.05|    2462.4|
|   10252|        33|    

In [ ]:
# Q17

In [27]:
df_products_clean = df_products_clean.withColumn("en_stock",col("units_in_stock")>0)

In [28]:
df_products_clean.show()

+----------+--------------------+-----------+-----------+--------------------+----------+--------------+--------------+-------------+------------+--------+
|product_id|        product_name|supplier_id|category_id|   quantity_per_unit|unit_price|units_in_stock|units_on_order|reorder_level|discontinued|en_stock|
+----------+--------------------+-----------+-----------+--------------------+----------+--------------+--------------+-------------+------------+--------+
|         1|                Chai|          8|          1|  10 boxes x 30 bags|      18.0|            39|             0|           10|           1|    true|
|         2|               Chang|          1|          1|  24 - 12 oz bottles|      19.0|            17|            40|           25|           1|    true|
|         3|       Aniseed Syrup|          1|          2| 12 - 550 ml bottles|      10.0|            13|            70|           25|           0|    true|
|         4|Chef Anton's Caju...|          2|          2|      4

In [29]:
df_orders_clean = df_orders_clean.withColumn("is_shipped",col("shipped_date").isNotNull())

In [30]:
df_orders_clean.show()

+--------+-----------+-----------+----------+-------------+------------+----------+-------+--------------------+--------------------+--------------+-----------+----------------+------------+----------+
|order_id|customer_id|employee_id|order_date|required_date|shipped_date|shipper_id|freight|           ship_name|        ship_address|     ship_city|ship_region|ship_postal_code|ship_country|is_shipped|
+--------+-----------+-----------+----------+-------------+------------+----------+-------+--------------------+--------------------+--------------+-----------+----------------+------------+----------+
|   10248|      VINET|          5|1996-07-04|   1996-08-01|  1996-07-16|         3|  32.38|Vins et alcools C...|  59 rue de l'Abbaye|         Reims|       NULL|           51100|      France|      true|
|   10249|      TOMSP|          6|1996-07-05|   1996-08-16|  1996-07-10|         1|  11.61|  Toms Spezialit�ten|       Luisenstr. 48|       M�nster|       NULL|           44087|     Germany|  

In [31]:
df_orders_clean.filter(col("is_shipped")==False).show()

+--------+-----------+-----------+----------+-------------+------------+----------+-------+--------------------+--------------------+---------------+-------------+----------------+------------+----------+
|order_id|customer_id|employee_id|order_date|required_date|shipped_date|shipper_id|freight|           ship_name|        ship_address|      ship_city|  ship_region|ship_postal_code|ship_country|is_shipped|
+--------+-----------+-----------+----------+-------------+------------+----------+-------+--------------------+--------------------+---------------+-------------+----------------+------------+----------+
|   11008|      ERNSH|          7|1998-04-08|   1998-05-06|        NULL|         3|  79.46|        Ernst Handel|        Kirchgasse 6|           Graz|         NULL|            8010|     Austria|     false|
|   11019|      RANCH|          6|1998-04-13|   1998-05-11|        NULL|         3|   3.17|       Rancho grande|Av. del Libertado...|   Buenos Aires|         NULL|            1010|

In [ ]:
# Q18

In [32]:
from pyspark.sql.functions import count

In [33]:
df_customers_clean.groupBy("customer_id").agg(count("*").alias("nb")).filter("nb > 1").show()

+-----------+---+
|customer_id| nb|
+-----------+---+
+-----------+---+



In [34]:
df_customers_clean.groupBy("customer_id").agg(count("*").alias("nb")).filter("nb = 1").show()

+-----------+---+
|customer_id| nb|
+-----------+---+
|      WOLZA|  1|
|      MAISD|  1|
|      BLAUS|  1|
|      MAGAA|  1|
|      FOLKO|  1|
|      ANATR|  1|
|      ISLAT|  1|
|      VAFFE|  1|
|      BLONP|  1|
|      CENTC|  1|
|      SPLIR|  1|
|      TRAIH|  1|
|      LILAS|  1|
|      WARTH|  1|
|      FRANR|  1|
|      SEVES|  1|
|      EASTC|  1|
|      HILAA|  1|
|      PARIS|  1|
|      HANAR|  1|
+-----------+---+
only showing top 20 rows



In [35]:
df_customers_clean.select(col("customer_id")).count()

91

In [36]:
df_customers_clean.select(col("customer_id")).distinct().count()

91

In [37]:
df_categories.show(3)

+-----------+-------------+--------------------+-------+
|category_id|category_name|         description|picture|
+-----------+-------------+--------------------+-------+
|          1|    Beverages|Soft drinks, coff...|     \x|
|          2|   Condiments|Sweet and savory ...|     \x|
|          3|  Confections|Desserts, candies...|     \x|
+-----------+-------------+--------------------+-------+
only showing top 3 rows



In [38]:
med_price

19.5

In [ ]:
# Q19

In [39]:
from pyspark.sql.functions import year

In [40]:
df_orders_clean = df_orders_clean.filter(year(col("order_date"))==1997)

In [41]:
df_products_clean = df_products_clean.filter((col("en_stock")) & (col("discontinued") == 0))

In [ ]:
# Q20

In [42]:
from pyspark.sql.functions import concat, lit

In [43]:
df_employees_clean = df_employees.select("employee_id","first_name","last_name","title","hire_date","city","country").withColumn("full_name",concat(col("first_name"),lit(" "),col("last_name")))

In [44]:
df_employees_clean.show()

+-----------+----------+---------+--------------------+----------+--------+-------+----------------+
|employee_id|first_name|last_name|               title| hire_date|    city|country|       full_name|
+-----------+----------+---------+--------------------+----------+--------+-------+----------------+
|          1|     Nancy|  Davolio|Sales Representative|1992-05-01| Seattle|    USA|   Nancy Davolio|
|          2|    Andrew|   Fuller|Vice President, S...|1992-08-14|  Tacoma|    USA|   Andrew Fuller|
|          3|     Janet|Leverling|Sales Representative|1992-04-01|Kirkland|    USA| Janet Leverling|
|          4|  Margaret|  Peacock|Sales Representative|1993-05-03| Redmond|    USA|Margaret Peacock|
|          5|    Steven| Buchanan|       Sales Manager|1993-10-17|  London|     UK| Steven Buchanan|
|          6|   Michael|   Suyama|Sales Representative|1993-10-17|  London|     UK|  Michael Suyama|
|          7|    Robert|     King|Sales Representative|1994-01-02|  London|     UK|     Rob

In [50]:
df_orders_clean.write.parquet("../data/tmp/orders")

AnalysisException: [PATH_ALREADY_EXISTS] Path file:/home/jovyan/data/tmp/orders already exists. Set mode as "overwrite" to overwrite the existing path.

In [46]:
df_products_clean.write.parquet("../data/tmp/products")

In [47]:
df_customers_clean.write.parquet("../data/tmp/customers")

In [48]:
df_employees_clean.write.parquet("../data/tmp/employees")

In [49]:
df_order_details_clean.write.parquet("../data/tmp/order_details")

In [3]:
df_categories.write.parquet("../data/tmp/categories")

In [4]:
df_shippers.write.parquet("../data/tmp/shippers")

In [5]:
df_suppliers.write.parquet("../data/tmp/suppliers")